# Lecture 06: Statistical Analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JamesMTucker/DATA_340_NLP/blob/master/Notebooks/Lecture_06_2023_02_14.ipynb)

Objective:

* Upcoming Assignments
* Some notes about this weeks reading
* Review Shannon's Entropy (Scipy package)
* Term-frequency inverse document frequency
* GitHub Questions

## Review of Shannon Entropy: Lecture 05 Notebook

[Open](Lecture_05_2023_02_09.ipynb#Shannon's-Entropy)

## Readings

### Martin & Jurafsky

[Link](https://github.com/JamesMTucker/DATA_340_NLP/blob/master/course_readings/Jurafsky_Martin_chapter_2_12-38.pdf)

### Grimmer et al.

[Link](https://github.com/JamesMTucker/DATA_340_NLP/blob/master/course_readings/Grimmer-Grimmer_48-62.pdf)

## TF-IDF: Term-Frequency Inverse Document Frequency

We learned from our analysis of Zipf's law that the most frequently occuring terms (bag of words) offer little information about how topics are discussed. We can get some idea of what topics might be discussed by the frequencies of a given term. But we often want and need to do more to analyze a document and its language. So, we created N-grams. We saw that an N-gram methodology provides more information about how language was used in a document, and we discussed how we could use entropy to analyse the bigrams in relationship to all the bigrams in a document. In addition to entropy, there are other useful metrics to extract a document's relevant terms: Term-frequency inverse document frequency and Pointwise Mutual Information.

### Informal Definition

General intuition of tf-idf is that words isolated to one particular text provide information as to how it relates to other documents in the corpus.

It is import to note the following definitions:

* `Document` = a news article, journal article, tweet, reddit post, etc.
* `Document vocabulary` = frequency of terms in a document
* `Corpus` = a collection of documents
* `Corpus Vocabulary` = frequency of terms in a corpus of documents

Thus,

`tf-idf` = `term_frequency` * `inverse_document_frequency`

`term_frequency` = count of a words appearence in a document
`inverse_document_frequency` = log(total_number_of_documents / number_of_documents_with_term) + 1

### Formal Definition

$$idf_{t} = log_{10}(\frac{N}{df_t})$$

`term frequency` = $tf_{t,d} = log_{10}(count(t,d) + 1)$ -- The intuition is that a word appearing 100 times does not make that word 100x more likely to be relevant to the meanining of the document. Therefore, we give a heigher weight to words that occur only in a few documents.

`total documents in collection` = $idf_{t} = log_{10}(\frac{N}{df_t})$ -- We let _N_ be the total number of documents in the corpus

### Case Study I

#### Corpus Creation

In [ ]:
# We will use a database of conversations about Eczema
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# If you are running this notebook in Google colab, uncomment this line of code and run
# from google.colab import drive
# drive.mount('/content/gdrive/', force_remount=True)
# file = Path('gdrive/MyDrive/DATA_340_3_NLP/Datasets/eczema.csv')
# convos = pd.read_csv(file)

convos = pd.read_csv('./data/eczema.csv')

In [ ]:
# Let's see the general data types used
convos.info()

In [ ]:
# Let's Random Sample some conversations
convos.sample(n=10)

In [ ]:
# Let's get an idea of the data size
convos.shape

In [ ]:
# Let's work with a subset
random_seed = 42
convos = convos.sample(n=100, random_state=1)
convos

#### Exploratory Data Analysis

Let's get more familiar with our data

In [ ]:
# Let's get an understanding of the length of our conversations
convos['word_total'] = convos['text'].apply(lambda x: len(x.split(' ')))
convos

In [ ]:
# Our sample distribution conversation length
(convos
 .drop(columns=['docID', 'text'])
 .plot(kind='hist', bins=50)
)

In [ ]:
# We can find the shortest and longest conversation using the min and max methods
convos['word_total'].max(), convos['word_total'].min()

In [ ]:
# Explore the longest and shortest
convos[convos['word_total'] == convos['word_total'].max()]['text'].values

In [ ]:
convos.query('word_total > 300')

In [ ]:
f"The 95th percentile of the tweet lenght is: {np.percentile(convos['word_total'].tolist(), 95)} words"

#### Standardize our corpus

Let's analyze the above two datasets using TF-IDF by doing the following.

* Tokenize the conversation
* We can try stemming the tweet and not-stemming to see the difference
* Count the freqeuncy of words in each tweet

In [ ]:
# Tokenize the tweet using SpaCy
import spacy
import string

from tqdm import tqdm
tqdm.pandas()

punctuation = list(string.punctuation)
NLP = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

In [ ]:
# Let's get a list of the lemmas to count the frequency of words but remove the punctuation
convos['tokens'] = convos['text'].progress_apply(lambda x: [x.lemma_.lower() for x in NLP(x) if x.lemma_.lower() not in punctuation])

In [ ]:
convos

In [ ]:
# Unwind the data on the tokens
convos_tokens = (convos
                  .explode('tokens')
                  .drop(columns=['text', 'word_total'])
                )

In [ ]:
convos_tokens

#### Word frequency

In [ ]:
# create a word frequency dataframe
term_frequency = (convos_tokens
                  .groupby(by=['docID', 'tokens'])
                  .agg({'tokens': 'count'})
                  .rename(columns={'tokens': 'term_frequency'})
                  .reset_index()
                  .rename(columns={'tokens': 'term'})
                 )

In [ ]:
term_frequency

In [ ]:
stop_words = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers',
         'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are',
         'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
         'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down',
         'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
         'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 've', 'll', 'amp']

In [ ]:
# remove stop words
term_frequency = term_frequency.drop(term_frequency[term_frequency['term'].isin(stop_words)].index)
term_frequency

#### Document frequency

In [ ]:
# Document Frequency
document_frequency = (term_frequency
                      .groupby(['docID', 'term'])
                      .size()
                      .unstack()
                      .sum()
                      .reset_index()
                      .rename(columns={0: 'document_frequency'})
                     )

In [ ]:
document_frequency

In [ ]:
# merge the document freqs into the term dataframe
term_frequency = term_frequency.merge(document_frequency)

#### Corpus count

In [ ]:
documents_in_corpus = term_frequency['docID'].nunique()
documents_in_corpus

In [ ]:
# inverse document frequency
term_frequency['idf'] = np.log((1 + documents_in_corpus) / (1 + term_frequency['document_frequency'])) + 1

In [ ]:
term_frequency

#### TF-IDF Calculation

In [ ]:
term_frequency['tfidf'] = term_frequency['term_frequency'] * term_frequency['idf']
term_frequency.sort_values(by=['term_frequency'], ascending=False)

#### Normalize with sklearn

In [ ]:
from sklearn import preprocessing
term_frequency['tfidf_norm'] = preprocessing.normalize(term_frequency[['tfidf']], axis=0, norm='l2')

In [ ]:
term_frequency

In [ ]:
top_n_terms = term_frequency.sort_values(by=['docID', 'tfidf'], ascending=[True, False]).groupby(['docID']).head(2)

In [ ]:
top_n_terms.head(10)

In [ ]:
docIds = top_n_terms['docID'].tolist()

In [ ]:
convos[convos['docID'] == 5924]['text'].values

### Case Study II

In [ ]:
# We can use sklearn
# R - https://cran.r-project.org/web/packages/superml/vignettes/Guide-to-TfidfVectorizer.html

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
convos = pd.read_csv('./data/eczema.csv')
convos.info()

In [ ]:
random_seed = 42
convos = convos.sample(n=100, random_state=1)
convos

#### Calculate tf-idf


In [ ]:
# Create an instance of the Tfidf vectorizer with stopwords

tfidf_vectorizer = TfidfVectorizer(stop_words='english')

#### Fit our model to the data

In [ ]:
# Fit on our model

vectors = tfidf_vectorizer.fit_transform(convos['text'])

#### Convert our trained model to a dataframe

In [ ]:
# Create dataframe with terms and tfidf
tfidf_df = pd.DataFrame(vectors.toarray(), index=convos.index.values, columns=tfidf_vectorizer.get_feature_names_out())
# tfidf_df.loc['00_Document Frequency'] = (tfidf_df > 0).sum()

In [ ]:
tfidf_vectorizer.get_feature_names_out().tolist()

#### Explore some selected terms

In [ ]:
# Explore some selected terms

tfidf_df[['eczema', 'lotion', 'skin', 'sleep', 'pain', 'zyrtec', 'afraid', 'dehydration', 'dermatologist']]

#### Create Heatmap of TF-IDF Terms in our data

In [ ]:
# Let's create a heatmap of higest terms in some of the documents

tfidf_df = (tfidf_df
            .stack()
            .reset_index()
            .rename(columns={0: 'tfidf', 'level_0': 'docID', 'level_1': 'term'})
           )

In [ ]:
# Group the documents by their n highest performing terms

n = 2
top_tfidf = (tfidf_df
             .sort_values(by=['docID','tfidf'], ascending=[True,False])
             .groupby(['docID'])
             .head(n)
            )

In [ ]:
top_tfidf

In [ ]:
#!pip install altair

In [ ]:
# Import altair for graphing the n highest terms in a heatmap

import altair as alt

# adding a little randomness to break ties in term ranking
top_tfidf_rand = top_tfidf.copy()
top_tfidf_rand['tfidf'] = top_tfidf_rand['tfidf'] + np.random.rand(top_tfidf.shape[0])*0.0001

base = alt.Chart(top_tfidf_rand).encode(
    x = 'rank:O',
    y = 'docID:N'
).transform_window(
    rank = "rank()",
    sort = [alt.SortField("tfidf", order="descending")],
    groupby = ["docID"],
)

# heatmap specification
heatmap = base.mark_rect().encode(
    color = 'tfidf:Q'
)

# text labels, white for darker heatmap colors
text = base.mark_text(baseline='middle').encode(
    text = 'term:N',
    color = alt.condition(alt.datum.tfidf >= 0.23, alt.value('white'), alt.value('black'))
)

# display the three superimposed visualizations
(heatmap + text).properties(width = 800)

In [ ]:
# Let's read one
convos.loc[105467]['text']

## GitHub Questions